# AttnRes VLM Ablation on Official CLEVR / CLEVR-CoGenT (Colab / CUDA)

Compute-constrained evaluation of baseline / Full / Block AttnRes VLMs on **fixed subsets** of the official CLEVR v1.0 and CLEVR-CoGenT v1.0 benchmarks.

- Official downloads only (`dl.fbaipublicfiles.com`)
- Resume-safe archive downloads + selective image extraction
- Artifacts under `MyDrive/SharedColab/attnres_vlm_ablation`
- Prior TinyShapes runs are preserved; TinyShapes is smoke-only now
- Reported CLEVR "test" = held-out subset of official validation (official test answers unused)
- CoGenT test = Condition B validation generalization

W&B project: `attnres-next-scale-vlm`.

**Note:** Answer supervision is at the `<answer>` marker (next-token), so runs use a new config hash and do not resume the leaky 100%-accuracy checkpoints.


In [6]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%cd /content
!rm -rf /content/AttnResGPT-next-scale
!git clone https://github.com/AtinChing/AttnResGPT-next-scale.git /content/AttnResGPT-next-scale
%cd /content/AttnResGPT-next-scale
!pip install -q -r requirements.txt
!git rev-parse --short HEAD
!git log -1 --oneline


/content
Cloning into '/content/AttnResGPT-next-scale'...
remote: Enumerating objects: 728, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 728 (delta 113), reused 155 (delta 68), pack-reused 513 (from 1)
Receiving objects: 100% (728/728), 34.93 MiB | 17.23 MiB/s, done.
Resolving deltas: 100% (406/406), done.
/content/AttnResGPT-next-scale
7da9f48
7da9f48 (HEAD -> main, origin/main, origin/HEAD) Switch VLM AttnRes Colab ablation to official CLEVR and CoGenT subsets.


In [8]:
import sys
from pathlib import Path

REPO = Path("/content/AttnResGPT-next-scale")
DRIVE_ROOT = Path("/content/drive/MyDrive")
SHARED_COLAB_ROOT = DRIVE_ROOT / "SharedColab"
PROJECT_ROOT = SHARED_COLAB_ROOT / "attnres_vlm_ablation"

SHARED_COLAB_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for name in [
    "source", "configs", "checkpoints", "runs", "logs", "metrics",
    "summaries", "plots", "tables", "cache", "manifests", "data",
]:
    (PROJECT_ROOT / name).mkdir(parents=True, exist_ok=True)

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.vlm.clevr.official import CLEVR_V1_NO_IMAGES
assert "dl.fbaipublicfiles.com" in CLEVR_V1_NO_IMAGES.url
print("REPO:", REPO)
print("cloned commit:", __import__("subprocess").check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())
print("DRIVE_ROOT:", DRIVE_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", PROJECT_ROOT / "data")


REPO: /content/AttnResGPT-next-scale
cloned commit: 7da9f48
DRIVE_ROOT: /content/drive/MyDrive
PROJECT_ROOT: /content/drive/MyDrive/SharedColab/attnres_vlm_ablation
DATA_ROOT: /content/drive/MyDrive/SharedColab/attnres_vlm_ablation/data


## Weights & Biases login


In [9]:
import os
import wandb
os.environ["WANDB_API_KEY"] = "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"

try:
    from google.colab import userdata
    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale-vlm"
WANDB_ENABLED = True
WANDB_MODE = "auto"
WANDB_RESUME = "allow"

wandb.login()
print("W&B project:", WANDB_PROJECT)
print("W&B entity:", WANDB_ENTITY)


wandb: Currently logged in as: atin5551 (atin5551-uc-davis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B project: attnres-next-scale-vlm
W&B entity: atin5551-uc-davis


## Configuration


In [10]:
BENCHMARK_MODE = "quick"  # "smoke", "quick", or "full"
RUN_STANDARD_CLEVR = True
RUN_COGENT = True

RESUME = True
FORCE_RESTART = False

# Initial pipeline check: Full variants only. Set False to include Block variants.
RUN_PRIMARY_FULL_ONLY = True
RUN_PRIMARY_FULL_GRID = True
RUN_BLOCK_GRID = False

SEEDS = [0, 1, 2]

STANDARD_VARIANTS = [
    "baseline",
    "encoder_full",
    "decoder_full",
    "both_full",
    "encoder_block",
    "decoder_block",
    "both_block",
]
PRIMARY_VARIANTS = ["baseline", "encoder_full", "decoder_full", "both_full"]
BLOCK_VARIANTS = ["encoder_block", "decoder_block", "both_block"]

# T4-friendly defaults; reduce BATCH_SIZE / raise GRAD_ACCUM_STEPS if OOM.
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2
CHECKPOINT_INTERVAL = 100
EVALUATION_INTERVAL = 1
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
MIXED_PRECISION = True
AMP_DTYPE = "auto"

VISION_D_MODEL = 128
VISION_N_LAYERS = 10
VISION_N_HEADS = 4
VISION_D_FF = 512
DECODER_D_MODEL = 128
DECODER_N_LAYERS = 10
DECODER_N_HEADS = 4
DECODER_D_FF = 512
NUM_BLOCKS = 5
DROPOUT = 0.0
IMAGE_SIZE = 128
PATCH_SIZE = 16

print("BENCHMARK_MODE:", BENCHMARK_MODE)
print("RUN_STANDARD_CLEVR:", RUN_STANDARD_CLEVR)
print("RUN_COGENT:", RUN_COGENT)
print("RUN_PRIMARY_FULL_ONLY:", RUN_PRIMARY_FULL_ONLY)
print("SEEDS:", SEEDS)
print("WANDB_PROJECT:", WANDB_PROJECT)
print("NOTE: CLEVR image archives are ~18GB and CoGenT ~24GB; downloads resume after disconnect.")


BENCHMARK_MODE: quick
RUN_STANDARD_CLEVR: True
RUN_COGENT: True
RUN_PRIMARY_FULL_ONLY: True
SEEDS: [0, 1, 2]
WANDB_PROJECT: attnres-next-scale-vlm
NOTE: CLEVR image archives are ~18GB and CoGenT ~24GB; downloads resume after disconnect.


## CUDA enforcement


In [11]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Select a Colab GPU runtime and rerun. "
        "Training will not fall back to CPU or MPS."
    )

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", props.name)
print(f"VRAM free/total: {free_bytes/1e9:.2f} / {total_bytes/1e9:.2f} GB")


GPU: Tesla T4
VRAM free/total: 15.53 / 15.64 GB


## Run experiment

This cell downloads official CLEVR/CoGenT archives into `PROJECT_ROOT/data`, builds fixed subset manifests, extracts only needed images, then trains the selected grids.


In [ ]:
from src.vlm.ablation.config import AblationExperimentConfig, config_hash, resolve_experiment_config
from src.vlm.ablation.runner import run_ablation_experiment

config = AblationExperimentConfig(
    benchmark_mode=BENCHMARK_MODE,
    run_mode=BENCHMARK_MODE,
    run_standard_clevr=RUN_STANDARD_CLEVR,
    run_cogent=RUN_COGENT,
    resume=RESUME,
    force_restart=FORCE_RESTART,
    run_primary_full_only=RUN_PRIMARY_FULL_ONLY,
    run_primary_full_grid=RUN_PRIMARY_FULL_GRID,
    run_block_grid=RUN_BLOCK_GRID and not RUN_PRIMARY_FULL_ONLY,
    seeds=list(SEEDS),
    primary_variants=list(PRIMARY_VARIANTS),
    block_variants=list(BLOCK_VARIANTS),
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    evaluation_interval=EVALUATION_INTERVAL,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=MIXED_PRECISION,
    amp_dtype=AMP_DTYPE,
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    vision_d_model=VISION_D_MODEL,
    vision_n_layers=VISION_N_LAYERS,
    vision_n_heads=VISION_N_HEADS,
    vision_d_ff=VISION_D_FF,
    decoder_d_model=DECODER_D_MODEL,
    decoder_n_layers=DECODER_N_LAYERS,
    decoder_n_heads=DECODER_N_HEADS,
    decoder_d_ff=DECODER_D_FF,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    project_root=str(PROJECT_ROOT),
    wandb_enabled=WANDB_ENABLED,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_mode=WANDB_MODE,
    wandb_resume=WANDB_RESUME,
)

preview = resolve_experiment_config(config)
print("Pre-download config hash preview:", config_hash(preview))
print("Model: vision/decoder", preview.vision_n_layers, preview.decoder_n_layers, "blocks", preview.num_blocks)
print("Image/patch:", preview.image_size, preview.patch_size)

summary = run_ablation_experiment(
    config,
    project_root=PROJECT_ROOT,
    source_root=REPO,
)
summary


Pre-download config hash preview: 35dba4701eb77b4c
Model: vision/decoder 10 10 blocks 5
Image/patch: 128 16
{
  "gpu_name": "Tesla T4",
  "cuda_version": "12.8",
  "torch_version": "2.11.0+cu128",
  "total_memory_bytes": 15637086208,
  "available_memory_bytes": 15527116800,
  "amp_dtype": "torch.float16"
}
[download] CLEVR_v1.0_no_images.zip: 9% (8388608/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 18% (16777216/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 28% (25165824/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 37% (33554432/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 46% (41943040/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 56% (50331648/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 65% (58720256/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 75% (67108864/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 84% (75497472/89363837 bytes)
[download] CLEVR_v1.0_no_images.zip: 93% (83886080/89363837 bytes)
[download] CLEVR_v1.0_no

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/learning_rate,▁▂▃▆██████████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃
train/loss,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▇▇▇▇█████
val/answer_token_nll,█▂▂▂▂▁▁▁▁▁
val/category_accuracy/attribute_comparison,▁▁▁▁▁▁▁▁▁▁
val/category_accuracy/attribute_query,▁▁▁▁▁▁▁▁▁▁
val/category_accuracy/counting,▁▇▇▇▇█████
val/category_accuracy/existence,▁▁▁▁▁▁▁▁▁▁
val/category_accuracy/integer_comparison,▁▁▁▁▁▁▁▁▁▁
+1,...


## Completion summary


In [ ]:
import json

summary_path = PROJECT_ROOT / "summaries" / "experiment_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Mounted Drive root:", DRIVE_ROOT)
print("Experiment root:", PROJECT_ROOT)
print("Repo clone:", REPO)
print("GPU:", json.dumps(summary.get("cuda", {}), indent=2))
print("Benchmark mode:", summary.get("benchmark_mode"))
print("Failures:", summary.get("failures"))
for benchmark, payload in (summary.get("benchmarks") or {}).items():
    print("=" * 60)
    print("Benchmark:", benchmark)
    print("  dataset_version:", payload.get("dataset_version"))
    print("  config_hash:", payload.get("config_hash"))
    print("  subset_manifest:", payload.get("subset_manifest_path"))
    print("  subset_hash:", payload.get("subset_manifest_hash"))
    print("  subset_sizes:", payload.get("subset_sizes"))
    print("  test_label:", payload.get("test_label"))
    print("  completed:", payload.get("completed"))
    print("  resumed:", payload.get("resumed"))
    print("  failed:", payload.get("failed"))
    print("  plots:", payload.get("plots_dir"))
    print("  tables:", payload.get("tables"))
    print("  Best by variant:")
    for variant, metrics in (payload.get("best_by_variant") or {}).items():
        print(
            f"    {variant}: val={metrics.get('validation_accuracy')} "
            f"test={metrics.get('test_accuracy')} "
            f"A->B drop={metrics.get('a_to_b_accuracy_drop')} "
            f"ckpt={metrics.get('checkpoint_best')}"
        )
assert str(PROJECT_ROOT).startswith(str(DRIVE_ROOT))
print("TinyShapes prior results remain under older runs/*/seed_*/<old_hash> directories.")
print("Official CLEVR/CoGenT runs live under runs/clevr_v1 and runs/clevr_cogent_v1.")


: 